# Factor Reweight Model Training

Train factor reweight models using:
1. **XGBoost** - Gradient boosted trees
2. **Ridge** - L2 regularized linear regression
3. **ElasticNet** - L1 + L2 regularized linear regression

## 1. Setup and Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Import our dataloader
from dataloader import FactorDataLoader

print("Libraries loaded successfully!")

In [ ]:
# Load data using config
loader = FactorDataLoader.from_config('config.yaml')
print(loader)

X, y = loader.load()

print(f"\nData loaded:")
print(f"  X shape: {X.shape}")
print(f"  y shape: {y.shape}")
print(f"  X memory: {X.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

## 2. Data Preprocessing

In [ ]:
# Check data quality
print("Data Quality Check:")
print(f"  X NaN count: {X.isna().sum().sum()}")
print(f"  X Inf count: {np.isinf(X.values).sum()}")
print(f"  y NaN count: {y.isna().sum()}")

# Handle NaN/Inf in features (fill with 0 or median)
X_clean = X.fillna(0)
X_clean = X_clean.replace([np.inf, -np.inf], 0)

# Get dates for time-series split
dates = X_clean.index.get_level_values('date')
unique_dates = dates.unique().sort_values()
print(f"\nUnique dates: {len(unique_dates)}")
print(f"Date range: {unique_dates.min()} to {unique_dates.max()}")

In [ ]:
# Time-series train/test split (use last 20% of dates as test)
split_idx = int(len(unique_dates) * 0.8)
train_dates = unique_dates[:split_idx]
test_dates = unique_dates[split_idx:]

print(f"Train dates: {train_dates.min()} to {train_dates.max()} ({len(train_dates)} days)")
print(f"Test dates:  {test_dates.min()} to {test_dates.max()} ({len(test_dates)} days)")

# Split data
train_mask = dates.isin(train_dates)
test_mask = dates.isin(test_dates)

X_train, X_test = X_clean[train_mask], X_clean[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"\nTrain set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")

In [ ]:
# Standardize features (important for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled: mean={X_train_scaled.mean():.6f}, std={X_train_scaled.std():.6f}")

## 3. Model Training

### 3.1 Evaluation Metrics

In [ ]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    """Calculate and print evaluation metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Information Coefficient (IC) - correlation between pred and actual
    ic = np.corrcoef(y_true, y_pred)[0, 1]
    
    print(f"\n{model_name} Results:")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  MAE:  {mae:.6f}")
    print(f"  R²:   {r2:.6f}")
    print(f"  IC:   {ic:.6f}")
    
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'ic': ic}

# Store results for comparison
results = {}

### 3.2 Ridge Regression (L2 Regularization)

In [ ]:
%%time
# Ridge Regression
# alpha: regularization strength (higher = more regularization)

ridge_model = Ridge(alpha=1.0, fit_intercept=True)
ridge_model.fit(X_train_scaled, y_train)

# Predict
y_pred_ridge_train = ridge_model.predict(X_train_scaled)
y_pred_ridge_test = ridge_model.predict(X_test_scaled)

# Evaluate
print("=" * 50)
print("RIDGE REGRESSION")
print("=" * 50)
print("\nTrain Set:")
evaluate_model(y_train, y_pred_ridge_train, "Ridge (Train)")
print("\nTest Set:")
results['Ridge'] = evaluate_model(y_test, y_pred_ridge_test, "Ridge (Test)")

### 3.3 ElasticNet (L1 + L2 Regularization)

In [ ]:
%%time
# ElasticNet
# alpha: regularization strength
# l1_ratio: 0 = Ridge, 1 = Lasso, 0.5 = balanced

elasticnet_model = ElasticNet(alpha=0.01, l1_ratio=0.5, fit_intercept=True, max_iter=1000)
elasticnet_model.fit(X_train_scaled, y_train)

# Predict
y_pred_en_train = elasticnet_model.predict(X_train_scaled)
y_pred_en_test = elasticnet_model.predict(X_test_scaled)

# Evaluate
print("=" * 50)
print("ELASTICNET")
print("=" * 50)
print("\nTrain Set:")
evaluate_model(y_train, y_pred_en_train, "ElasticNet (Train)")
print("\nTest Set:")
results['ElasticNet'] = evaluate_model(y_test, y_pred_en_test, "ElasticNet (Test)")

# Check sparsity (how many coefficients are zero)
n_zero = np.sum(elasticnet_model.coef_ == 0)
print(f"\nSparsity: {n_zero}/{len(elasticnet_model.coef_)} coefficients are zero ({n_zero/len(elasticnet_model.coef_)*100:.1f}%)")

### 3.4 XGBoost (Gradient Boosted Trees)

In [ ]:
%%time
# XGBoost (no need for scaling - tree models are scale invariant)

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,  # L1 regularization
    'reg_lambda': 1.0,  # L2 regularization
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

xgb_model = xgb.XGBRegressor(**xgb_params)

# Use unscaled data for XGBoost
xgb_model.fit(
    X_train.values, y_train.values,
    eval_set=[(X_test.values, y_test.values)],
    verbose=False
)

# Predict
y_pred_xgb_train = xgb_model.predict(X_train.values)
y_pred_xgb_test = xgb_model.predict(X_test.values)

# Evaluate
print("=" * 50)
print("XGBOOST")
print("=" * 50)
print("\nTrain Set:")
evaluate_model(y_train, y_pred_xgb_train, "XGBoost (Train)")
print("\nTest Set:")
results['XGBoost'] = evaluate_model(y_test, y_pred_xgb_test, "XGBoost (Test)")

## 4. Model Comparison

In [ ]:
# Compare all models
results_df = pd.DataFrame(results).T
results_df = results_df.round(6)

print("=" * 60)
print("MODEL COMPARISON (Test Set)")
print("=" * 60)
print(results_df)

# Plot comparison
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

metrics = ['rmse', 'mae', 'r2', 'ic']
titles = ['RMSE (lower better)', 'MAE (lower better)', 'R² (higher better)', 'IC (higher better)']

for ax, metric, title in zip(axes, metrics, titles):
    results_df[metric].plot(kind='bar', ax=ax, color=['steelblue', 'coral', 'seagreen'])
    ax.set_title(title)
    ax.set_ylabel(metric.upper())
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Feature Importance (XGBoost)

In [ ]:
# XGBoost feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).plot(kind='barh', x='feature', y='importance', ax=ax, legend=False)
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importances (XGBoost)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Important Features:")
print(importance.head(10).to_string(index=False))

## 6. Save Models

In [ ]:
import joblib
from pathlib import Path

# Create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Save models
joblib.dump(ridge_model, models_dir / "ridge_model.pkl")
joblib.dump(elasticnet_model, models_dir / "elasticnet_model.pkl")
joblib.dump(xgb_model, models_dir / "xgboost_model.pkl")
joblib.dump(scaler, models_dir / "scaler.pkl")

print("Models saved to ./models/")
print(f"  - ridge_model.pkl")
print(f"  - elasticnet_model.pkl")
print(f"  - xgboost_model.pkl")
print(f"  - scaler.pkl")

## 7. Hyperparameter Tuning (Optional)

Example of cross-validation with TimeSeriesSplit for hyperparameter tuning.

In [ ]:
# Example: Grid search for Ridge alpha
from sklearn.model_selection import GridSearchCV

# Define parameter grid
ridge_params = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Grid search
ridge_grid = GridSearchCV(
    Ridge(fit_intercept=True),
    ridge_params,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

ridge_grid.fit(X_train_scaled, y_train)

print("Ridge Hyperparameter Tuning:")
print(f"  Best alpha: {ridge_grid.best_params_['alpha']}")
print(f"  Best CV score (neg MSE): {ridge_grid.best_score_:.6f}")

# Results
cv_results = pd.DataFrame({
    'alpha': ridge_params['alpha'],
    'mean_score': -ridge_grid.cv_results_['mean_test_score'],
    'std_score': ridge_grid.cv_results_['std_test_score']
})
print("\nCV Results:")
print(cv_results.to_string(index=False))